# LLM memory arithmetic, computed

This notebook computes the memory numbers behind the standard interview
questions, so you can check your whiteboard arithmetic:

- "How much memory does it take to train a 70B model? Walk me through it."
  (notes.md §2.1, §2.2, §2.4; exercise 4)
- "What is the KV cache for a 7B model at batch 32, 4k context?"
  (notes.md §5.1, §5.2)
- "How many sequences fit with MHA vs GQA on a 70B model?" (exercise 5)

Each number is a few multiplications. The cells first define the formulas, then
run four cases. Sizes are printed in GiB (2^30 bytes) and, in parentheses, in
GB (10^9 bytes); the notes use GB, and GPUs are sold in GB.

`uv run jupyter execute ml/07_llm_stack/memory_math.ipynb`

## Units and model configurations

Two model shapes cover most interview arithmetic: a 7B model shaped like
Llama-2-7B (full multi-head attention) and a 70B model shaped like Llama-2-70B
(grouped-query attention with 8 KV heads). The training-memory questions need
only the parameter count; the KV-cache questions also need layers, heads and
width.

In [1]:
GiB = 1024**3
GB = 1e9

LLAMA_7B = dict(name="7B (Llama-2-7B-like)", params=7e9,
                n_layers=32, n_heads=32, n_kv_heads=32, d_model=4096)
LLAMA_70B = dict(name="70B (Llama-2-70B-like)", params=70e9,
                 n_layers=80, n_heads=64, n_kv_heads=8, d_model=8192)


def fmt(b):
    return f"{b / GiB:8.1f} GiB ({b / GB:7.1f} GB)"


def table(title, rows, total_label="TOTAL"):
    width = 68
    print(f"\n{title}")
    print("-" * width)
    for k, v in rows.items():
        print(f"  {k:<38}{fmt(v)}")
    print("-" * width)
    print(f"  {total_label:<38}{fmt(sum(rows.values()))}")

## Training state: 16 bytes per parameter

Mixed precision with Adam stores, per parameter (notes.md §2.1):

```text
fp16 working weights   2 bytes   (fast tensor-core matmuls)
fp16 gradients         2 bytes   (backward output, at matmul precision)
fp32 master weights    4 bytes   (an fp16 update eta*g below half the fp16
                                  spacing is rounded away)
fp32 Adam m            4 bytes   (running mean of gradients)
fp32 Adam v            4 bytes   (running mean of squared gradients)
                      ---------
                      16 bytes/param   ("optimizer state" = the fp32 12)
```

ZeRO shards parts of this across the G data-parallel GPUs:

```text
stage 0 (plain DP):  16 * N                  on every GPU
stage 1:             (2+2) * N + 12 * N / G  (shard optimizer state)
stage 2:             2 * N + (2+12) * N / G  (+ shard gradients)
stage 3 (FSDP):      16 * N / G              (+ shard weights)
```

The cell below defines both formulas and prints bytes per parameter for a few
precision/optimizer combinations: mixed precision with Adam and pure fp32 with
Adam both come to 16, and SGD with momentum to 12. (Accumulating gradients in
fp32 across microbatches would add another 4, giving 20; that variant is not
coded here.)

In [2]:
def training_state_bytes(params, precision="mixed", optimizer="adam"):
    # per-component bytes for the full (unsharded) training state
    if precision == "mixed":
        w_bytes, g_bytes = 2, 2          # fp16/bf16 working copies
        master = 4                        # fp32 master weights
    elif precision == "fp32":
        w_bytes, g_bytes = 4, 4
        master = 0                        # weights are already fp32
    else:
        raise ValueError(precision)

    if optimizer == "adam":
        opt = master + 4 + 4              # master + m + v (all fp32)
    elif optimizer == "sgd_momentum":
        opt = master + 4                  # master + velocity
    elif optimizer == "sgd":
        opt = master
    else:
        raise ValueError(optimizer)

    return {
        "weights": params * w_bytes,
        "gradients": params * g_bytes,
        "optimizer state": params * opt,
    }


def zero_shard(state, G, stage):
    # ZeRO: divide the sharded components by the number of GPUs G
    sharded = {0: [], 1: ["optimizer state"],
               2: ["optimizer state", "gradients"],
               3: ["optimizer state", "gradients", "weights"]}[stage]
    return {k: (v / G if k in sharded else v) for k, v in state.items()}


for prec, opt in [("mixed", "adam"), ("fp32", "adam"),
                  ("mixed", "sgd_momentum"), ("fp32", "sgd_momentum")]:
    print(f"{prec:>6} + {opt:<13} {sum(training_state_bytes(1, prec, opt).values()):>3} bytes/param")
assert sum(training_state_bytes(1).values()) == 16

 mixed + adam           16 bytes/param
  fp32 + adam           16 bytes/param
 mixed + sgd_momentum   12 bytes/param
  fp32 + sgd_momentum   12 bytes/param


## Activation memory

The standard per-layer estimate (Korthikanti et al., "Reducing Activation
Recomputation in Large Transformer Models", eq. 2) for fp16, standard
attention, no sequence parallelism and no recomputation:

```text
bytes per layer ≈ s * b * h * (34 + 5 * a * s / h)
```

with s = sequence length, b = microbatch, h = hidden size, a = heads. The 34
counts the h-sized tensors saved for backward (layer-norm inputs, Q, K, V, MLP
intermediates at 4h, and so on); the `5 * a * s / h` term is the s×s attention
matrices (softmax input and output, dropout mask), which FlashAttention avoids
storing. With full gradient checkpointing we keep only each layer's fp16 input,
`2 * s * b * h` bytes, plus one layer's full activations while that layer is
recomputed during backward.

The cell prints the 70B numbers used in notes.md §2.2 (s = 4096, b = 1):
about 6.5 GB per layer, about 520 GB over 80 layers, and about 5.4 GB of
checkpointed layer inputs.

In [3]:
def activation_bytes_per_layer(seq, batch, d_model, n_heads):
    return seq * batch * d_model * (34 + 5 * n_heads * seq / d_model)


def activation_bytes(seq, batch, d_model, n_heads, n_layers, checkpointing=False):
    if checkpointing:
        # each layer's fp16 input, plus one layer's activations during recompute
        return (2 * seq * batch * d_model * n_layers
                + activation_bytes_per_layer(seq, batch, d_model, n_heads))
    return n_layers * activation_bytes_per_layer(seq, batch, d_model, n_heads)


m = LLAMA_70B
per_layer = activation_bytes_per_layer(4096, 1, m["d_model"], m["n_heads"])
inputs_only = 2 * 4096 * 1 * m["d_model"] * m["n_layers"]
print(f"70B, s=4096, b=1: per layer        {per_layer / GB:6.1f} GB")
print(f"                  all {m['n_layers']} layers     {m['n_layers'] * per_layer / GB:6.1f} GB")
print(f"                  layer inputs only {inputs_only / GB:6.1f} GB  (checkpointing)")
assert abs(per_layer / GB - 6.5) < 0.05 and abs(inputs_only / GB - 5.4) < 0.05

70B, s=4096, b=1: per layer           6.5 GB
                  all 80 layers      520.8 GB
                  layer inputs only    5.4 GB  (checkpointing)


## KV cache

For each token and each layer we cache one K vector and one V vector of size
`n_kv_heads * d_head` (notes.md §5.2):

```text
bytes = 2 (K and V) * n_layers * n_kv_heads * d_head * bytes_per_el * seq_len * batch
```

With full multi-head attention `n_kv_heads = n_heads`, so
`n_kv_heads * d_head = d_model`; grouped-query attention divides the cache by
`n_heads / n_kv_heads`. The cell defines the formula and prints the per-token
size for the 7B model, which should be 512 KiB (16 KiB per layer × 32 layers).

In [4]:
def kv_cache_bytes(seq, batch, n_layers, n_kv_heads, n_heads, d_model, bytes_per_el=2):
    d_head = d_model // n_heads
    per_token_per_layer = 2 * n_kv_heads * d_head * bytes_per_el  # K and V
    return per_token_per_layer * n_layers * seq * batch


m = LLAMA_7B
per_tok = kv_cache_bytes(1, 1, m["n_layers"], m["n_kv_heads"], m["n_heads"], m["d_model"])
print(f"7B KV cache per token: {per_tok / 1024:.0f} KiB")
assert per_tok == 512 * 1024

7B KV cache per token: 512 KiB


## Case 1: train a 7B model on one GPU?

Mixed precision with Adam at sequence length 4096 and microbatch 1. Expect
112 GB (104 GiB) of training state alone, which does not fit on an 80 GB GPU
even before activations. The fix is sharding: ZeRO-2 across 8 GPUs brings the
state to about 26 GB per GPU, the figure in notes.md §2.4.

In [5]:
m = LLAMA_7B
seq, batch = 4096, 1
print("=" * 70)
print("CASE 1: train a 7B model, mixed precision + Adam, seq 4096, microbatch 1")
print("=" * 70)
state = training_state_bytes(m["params"])
table("Training state (unsharded): 16 bytes/param", state)
act = activation_bytes(seq, batch, m["d_model"], m["n_heads"], m["n_layers"])
act_ckpt = activation_bytes(seq, batch, m["d_model"], m["n_heads"], m["n_layers"],
                            checkpointing=True)
table("Activations", {
    "no checkpointing (all layers)": act,
    "with full checkpointing": act_ckpt,
}, total_label="(alternatives, not summed)")
zero2 = sum(zero_shard(state, 8, 2).values())
print(f"\n  Verdict: {fmt(sum(state.values()))} of state alone; an 80 GB GPU holds "
      f"{80 * GB / GiB:.1f} GiB.")
print("  Even a 7B model does NOT train on one 80 GB GPU without sharding.")
print(f"  ZeRO-2 across 8 GPUs: {fmt(zero2)} of state per GPU, plus activations.")
assert sum(state.values()) == 112e9 and abs(zero2 / GB - 26.25) < 1e-9

CASE 1: train a 7B model, mixed precision + Adam, seq 4096, microbatch 1

Training state (unsharded): 16 bytes/param
--------------------------------------------------------------------
  weights                                   13.0 GiB (   14.0 GB)
  gradients                                 13.0 GiB (   14.0 GB)
  optimizer state                           78.2 GiB (   84.0 GB)
--------------------------------------------------------------------
  TOTAL                                    104.3 GiB (  112.0 GB)

Activations
--------------------------------------------------------------------
  no checkpointing (all layers)             97.0 GiB (  104.2 GB)
  with full checkpointing                    4.0 GiB (    4.3 GB)
--------------------------------------------------------------------
  (alternatives, not summed)               101.0 GiB (  108.5 GB)

  Verdict:    104.3 GiB (  112.0 GB) of state alone; an 80 GB GPU holds 74.5 GiB.
  Even a 7B model does NOT train on one 80 GB GPU

## Case 2: train a 70B model with ZeRO across 64 GPUs

The canonical walk-through. Unsharded state is 1.12 TB (weights 140 GB,
gradients 140 GB, optimizer 840 GB). Expect the per-GPU table from notes.md
§2.4: plain DP 1120 GB, ZeRO-1 about 293 GB, ZeRO-2 about 155 GB, ZeRO-3
17.5 GB. Only ZeRO-3 fits. The last table adds checkpointed activations for
one 4096-token sequence per GPU.

In [6]:
m = LLAMA_70B
G = 64
seq, batch = 4096, 1
print("=" * 70)
print(f"CASE 2: train a 70B model, ZeRO stages across {G} GPUs")
print("=" * 70)
state = training_state_bytes(m["params"])
table("Training state (unsharded): 16 bytes/param", state)

print(f"\nPer-GPU state under ZeRO (G = {G}):")
print("-" * 70)
print(f"  {'stage':<10}{'shards':<30}{'per-GPU state':>18}  {'fits 80GB?':>10}")
shard_desc = {0: "nothing (plain DP)", 1: "optimizer state (12B/p)",
              2: "+ gradients (2B/p)", 3: "+ weights (2B/p) [= FSDP]"}
per_gpu = {}
for stage in range(4):
    per_gpu[stage] = sum(zero_shard(state, G, stage).values())
    fits = "yes" if per_gpu[stage] < 75 * GiB else "NO"
    print(f"  ZeRO-{stage:<5}{shard_desc[stage]:<30}"
          f"{per_gpu[stage] / GB:>14.1f} GB  {fits:>10}")
assert [round(per_gpu[s] / GB, 1) for s in range(4)] == [1120.0, 293.1, 155.3, 17.5]

act_ckpt = activation_bytes(seq, batch, m["d_model"], m["n_heads"], m["n_layers"],
                            checkpointing=True)
table(f"\nZeRO-3 per-GPU total (seq {seq}, microbatch {batch}, full checkpointing)", {
    "sharded state (16 B/param / 64)": per_gpu[3],
    "activations (checkpointed)": act_ckpt,
})
print("  Add communication buffers and fragmentation in practice. ZeRO-3 moves")
print("  ~1.5x the data of plain DP (an extra weight all-gather in backward).")

CASE 2: train a 70B model, ZeRO stages across 64 GPUs

Training state (unsharded): 16 bytes/param
--------------------------------------------------------------------
  weights                                  130.4 GiB (  140.0 GB)
  gradients                                130.4 GiB (  140.0 GB)
  optimizer state                          782.3 GiB (  840.0 GB)
--------------------------------------------------------------------
  TOTAL                                   1043.1 GiB ( 1120.0 GB)

Per-GPU state under ZeRO (G = 64):
----------------------------------------------------------------------
  stage     shards                             per-GPU state  fits 80GB?
  ZeRO-0    nothing (plain DP)                    1120.0 GB          NO
  ZeRO-1    optimizer state (12B/p)                293.1 GB          NO
  ZeRO-2    + gradients (2B/p)                     155.3 GB          NO
  ZeRO-3    + weights (2B/p) [= FSDP]               17.5 GB         yes


ZeRO-3 per-GPU total (seq 4096

## Case 3: 7B inference at batch 32, 4k context

The canonical KV-cache question (notes.md §5.2). Expect 68.7 GB of cache
against 14 GB of fp16 weights, about 5× the model, for a total near 83 GB that
does not fit on an 80 GB A100. The cell then shows what GQA with 8 KV heads
and int8 KV storage save, and the batch-1 decode speed limit from memory
bandwidth (notes.md §5.1): about 143 tok/s in fp16 and about 570 tok/s with
int4 weights on a 2 TB/s A100.

In [7]:
m = LLAMA_7B
seq, batch = 4096, 32
print("=" * 70)
print("CASE 3: 7B inference, batch 32, context 4096 (KV-cache math)")
print("=" * 70)
d_head = m["d_model"] // m["n_heads"]
per_tok_layer = 2 * m["n_kv_heads"] * d_head * 2
per_tok = per_tok_layer * m["n_layers"]
print(f"\n  per token per layer: 2 (K,V) x {m['n_kv_heads']}x{d_head} x 2 B (fp16) "
      f"= {per_tok_layer / 1024:.0f} KiB")
print(f"  per token, {m['n_layers']} layers: {per_tok / 1024:.0f} KiB")

kv = kv_cache_bytes(seq, batch, m["n_layers"], m["n_kv_heads"], m["n_heads"], m["d_model"])
weights_fp16 = m["params"] * 2
table(f"Inference memory, batch {batch} @ {seq} ctx", {
    "weights (fp16)": weights_fp16,
    "KV cache (MHA, fp16)": kv,
})
print(f"\n  KV cache is {kv / weights_fp16:.1f}x the weights. Total "
      f"{(kv + weights_fp16) / GB:.0f} GB vs 80 GB on an A100: does NOT fit.")
assert round(kv / GB, 1) == 68.7

kv_gqa = kv_cache_bytes(seq, batch, m["n_layers"], 8, m["n_heads"], m["d_model"])
kv_gqa_int8 = kv_cache_bytes(seq, batch, m["n_layers"], 8, m["n_heads"], m["d_model"],
                             bytes_per_el=1)
table("The standard fixes", {
    "GQA, 8 KV heads (/4)": kv_gqa,
    "GQA + int8 KV (/8)": kv_gqa_int8,
}, total_label="(alternatives, not summed)")

bw = 2e12  # A100 HBM bandwidth, bytes/s
print("\n  Decode speed limit (A100, 2 TB/s): every token reads all weights once")
print(f"    batch 1, fp16: {bw / weights_fp16:6.0f} tok/s   (2e12 / {weights_fp16 / GB:.0f}e9)")
print(f"    batch 1, int4: {bw / (m['params'] * 0.5):6.0f} tok/s   "
      "(at batch 1, quantization raises speed, not only capacity)")

CASE 3: 7B inference, batch 32, context 4096 (KV-cache math)

  per token per layer: 2 (K,V) x 32x128 x 2 B (fp16) = 16 KiB
  per token, 32 layers: 512 KiB

Inference memory, batch 32 @ 4096 ctx
--------------------------------------------------------------------
  weights (fp16)                            13.0 GiB (   14.0 GB)
  KV cache (MHA, fp16)                      64.0 GiB (   68.7 GB)
--------------------------------------------------------------------
  TOTAL                                     77.0 GiB (   82.7 GB)

  KV cache is 4.9x the weights. Total 83 GB vs 80 GB on an A100: does NOT fit.

The standard fixes
--------------------------------------------------------------------
  GQA, 8 KV heads (/4)                      16.0 GiB (   17.2 GB)
  GQA + int8 KV (/8)                         8.0 GiB (    8.6 GB)
--------------------------------------------------------------------
  (alternatives, not summed)                24.0 GiB (   25.8 GB)

  Decode speed limit (A100, 2 TB

## Case 4: 70B KV cache, MHA vs GQA (exercise 5)

A 70B model with 80 layers, d_model 8192 and 64 heads of size 128, fp16 cache,
4096-token context. With full multi-head attention the cache is about 10.7 GB
per sequence; with 8 KV heads it is about 1.34 GB, 8× less. With int4 weights
(35 GB) on one 80 GB GPU and a few GB of overhead, about 40 GB is left for
cache: about 3 sequences with MHA versus about 29 with GQA.

In [8]:
m = LLAMA_70B
seq = 4096
kv_mha = kv_cache_bytes(seq, 1, m["n_layers"], m["n_heads"], m["n_heads"], m["d_model"])
kv_gqa = kv_cache_bytes(seq, 1, m["n_layers"], m["n_kv_heads"], m["n_heads"], m["d_model"])
free = 40 * GB
print("=" * 70)
print("CASE 4: 70B KV cache per 4096-token sequence, fp16")
print("=" * 70)
print(f"  MHA (64 KV heads): {kv_mha / GB:6.2f} GB per sequence, "
      f"{int(free // kv_mha):>3} sequences in 40 GB")
print(f"  GQA ( 8 KV heads): {kv_gqa / GB:6.2f} GB per sequence, "
      f"{int(free // kv_gqa):>3} sequences in 40 GB")
assert round(kv_mha / GB, 1) == 10.7 and round(kv_gqa / GB, 2) == 1.34
assert int(free // kv_mha) == 3 and int(free // kv_gqa) == 29
print("\nAll checks passed. Every number above takes at most a few multiplications;")
print("practice doing them on a whiteboard.")

CASE 4: 70B KV cache per 4096-token sequence, fp16
  MHA (64 KV heads):  10.74 GB per sequence,   3 sequences in 40 GB
  GQA ( 8 KV heads):   1.34 GB per sequence,  29 sequences in 40 GB

All checks passed. Every number above takes at most a few multiplications;
practice doing them on a whiteboard.
